In [ ]:
%pip install -q sentence-transformers

In [ ]:
import os
import pandas as pd
import torch
import re
import gc
from sentence_transformers import SentenceTransformer, util

In [ ]:

import re


def remove_emoji(text):
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F9FF"
        u"\U00002700-\U000027BF"
        u"\U0001FA00-\U0001FA6F"
        u"\U00002500-\U00002BFF"
        u"\U00002600-\U000026FF"
        u"✀-➿"
        u"⌀-⏿"
        "✓✗⚠►▸▼▲◄→←↑↓│├└┌┐┘┤┬┴┼"
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub(' ', text)


def replace_arrows(text):
    if not isinstance(text, str):
        return ''
    arrows = [
        ('-->', ' leads to '),
        ('=>',  ' leads to '),
        ('→',   ' leads to '),
        ('►',   ' leads to '),
        ('▼',   ' then '),
        ('▲',   ' then '),
        ('←',   ' comes from '),
    ]
    for arrow, replacement in arrows:
        text = text.replace(arrow, replacement)
    return text


def extract_code_block_content(text):
    def clean_block(match):
        code = match.group(1)
        code = ''.join(
            ' ' if (0x2500 <= ord(ch) <= 0x257F) else ch
            for ch in code
        )
        code = re.sub(r'[ \t]+', ' ', code)
        code = re.sub(r'\n{2,}', '\n', code)
        return code.strip()

    return re.sub(
        r'```(?:\w+)?\n?(.*?)```',
        clean_block,
        text,
        flags=re.DOTALL
    )


def convert_table_to_text_safe(text):
    lines = text.split('\n')
    result = []
    i = 0

    while i < len(lines):
        line = lines[i].strip()

        if line.startswith('|') and line.endswith('|'):
            if re.match(r'^\|[\s\-:]+\|$', line):
                i += 1
                continue

            cells = [c.strip() for c in line.strip('|').split('|') if c.strip()]

            if i + 1 < len(lines) and re.match(r'^\|[\s\-:|]+\|$', lines[i + 1].strip()):
                headers = cells
                i += 2

                while i < len(lines):
                    row = lines[i].strip()
                    if not (row.startswith('|') and row.endswith('|')):
                        break

                    values = [c.strip() for c in row.strip('|').split('|') if c.strip()]
                    headers = [re.sub(r'\*+(.+?)\*+', r'\1', h) for h in headers]
                    values  = [re.sub(r'\*+(.+?)\*+', r'\1', v) for v in values]

                    pairs = []
                    for h, v in zip(headers, values):
                        if h and v:
                            pairs.append(f"{h} is {v}")

                    if pairs:
                        result.append('. '.join(pairs) + '.')

                    i += 1
            else:
                result.append('. '.join(cells) + '.')
                i += 1
        else:
            result.append(lines[i])
            i += 1

    return '\n'.join(result)


def ensure_line_terminators(text):
    lines = text.split('\n')
    result = []
    for line in lines:
        stripped = line.strip()
        match = re.match(r'^(\-|\*|•|\d+[\.\)])\s+(.+)', stripped)
        if match:
            stripped = match.group(2).strip()
        if stripped and stripped[-1] not in '.!?:':
            stripped += '.'
        result.append(stripped)
    return '\n'.join(result)


def ensure_sentence_terminators(text):
    paragraphs = text.split('\n')
    result = []
    for para in paragraphs:
        para = para.strip()
        if not para:
            result.append('')
            continue
        doc = nlp(para)
        sentences = []
        for sent in doc.sents:
            s = sent.text.strip()
            if s and s[-1] not in '.!?:':
                s += '.'
            sentences.append(s)
        result.append(' '.join(sentences))
    return '\n'.join(result)

def remove_markdown_dividers(text):
    return re.sub(r'^\s*[-_*]{2,}\s*$', '', text, flags=re.MULTILINE)


def clean_markdown(text):



    text = replace_arrows(text)
    text = remove_emoji(text)

    text = extract_code_block_content(text)

    text = convert_table_to_text_safe(text)

    text = remove_markdown_dividers(text)


    text = re.sub(r'\*{1,3}(.+?)\*{1,3}', r'\1', text)
    text = re.sub(r'`(.+?)`', r'\1', text)
    text = re.sub(r'\[(.+?)\]\(.+?\)', r'\1', text)
    text = re.sub(r'^[\-\*\+]\s+', '', text, flags=re.MULTILINE)

   
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    text = text.replace('“', '"').replace('”', '"')
    text = text.replace('‘', "'").replace('’', "'")
    text = re.sub(r'"{2,}', '"', text)
    text = text.replace('"', '')
    text = text.replace('—', ' ')
    text = text.replace('–', ' ')
    text = re.sub(r'[ \t]+', ' ', text)  

    text = re.sub(r'#+\s*', '', text)
    


    return text.strip()

In [ ]:
INPUT_DIR  = "../Dataset/responses"
OUTPUT_DIR = "./results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Working dir:', os.getcwd())
print('Data dir   :', INPUT_DIR)
print('Output dir :', OUTPUT_DIR)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

sbert = SentenceTransformer('BAAI/bge-m3', device=device)

In [ ]:
def encode_in_batches(model, texts, batch_size=8):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        with torch.no_grad():
            embs = model.encode(batch, convert_to_tensor=True, show_progress_bar=False)
        all_embeddings.append(embs.cpu())
        torch.cuda.empty_cache()
    return torch.cat(all_embeddings, dim=0)


def run_sem_for_file(input_path, output_path,
                     compare_levels=None):
   
    df = pd.read_csv(input_path, sep='\t')
    df['LLM_Response'] = df['LLM_Response'].astype(str)
    df['Clean_Response'] = df['LLM_Response'].apply(clean_markdown)

    model_col = 'Model_Name' if 'Model_Name' in df.columns else 'Answer_Model'
    if compare_levels is None:
        compare_levels = ['High Literacy', 'Mid Literacy', 'Low Literacy']

    print(f" Loaded {len(df)} rows")
    print(f"Models: {df[model_col].unique()}")
    print(f"Literacy Levels: {df['Literacy_Level'].unique()}")

    results = []

    for model_name in df[model_col].unique():
        df_model = df[df[model_col] == model_name]

        hl_df = (df_model[df_model['Literacy_Level'] == 'High Literacy']
                 [['ID', 'Clean_Response']]
                 .rename(columns={'Clean_Response': 'HL_Text'}))

        if hl_df.empty:
            print(f"No High Literacy rows for model: {model_name} — skipping")
            continue

        for level in compare_levels:
            comp_df = (df_model[df_model['Literacy_Level'] == level]
                       [['ID', 'Clean_Response']]
                       .rename(columns={'Clean_Response': 'Comp_Text'}))

            pairs = pd.merge(hl_df, comp_df, on='ID')

            if pairs.empty:
                print(f"{model_name} - {level}: no matching pairs, skipping")
                continue

            hl_embs   = encode_in_batches(sbert, pairs['HL_Text'].tolist(),   batch_size=8)
            comp_embs = encode_in_batches(sbert, pairs['Comp_Text'].tolist(), batch_size=8)

            for i, (_, row) in enumerate(pairs.iterrows()):
                sim = util.cos_sim(hl_embs[i], comp_embs[i]).item()
                results.append({
                    model_col:             model_name,
                    'ID':                  row['ID'],
                    'Compared_Level':      level,
                    'Semantic_Similarity': round(sim, 4),
                    'Semantic_Loss':       round(1 - sim, 4)
                })

            print(f" {model_name} | High vs {level} — {len(pairs)} pairs done")

            del hl_embs, comp_embs
            gc.collect()
            torch.cuda.empty_cache()

    sem_df = pd.DataFrame(results)
    sem_df.to_csv(output_path, sep='\t', index=False)
    print(f"\n Saved: {output_path}\n")

    summary = sem_df.groupby([model_col, 'Compared_Level']).agg(
        Pairs          = ('Semantic_Similarity', 'count'),
        Avg_Similarity = ('Semantic_Similarity', 'mean'),
        Avg_Loss       = ('Semantic_Loss',       'mean')
    ).round(4)
    summary = summary.reindex(compare_levels, level='Compared_Level')

    print(f"{'='*60}")
    print("  SEMANTIC SIMILARITY: High vs Other Levels (BGE-M3)")
    print(f"{'='*60}")
    print(summary.to_string())

    return sem_df

In [ ]:
run_sem_for_file(
    input_path  = os.path.join(INPUT_DIR,  "explicitclaude.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "SemanticSimilarity_claude_explicit.tsv"),
)

In [ ]:
run_sem_for_file(
    input_path  = os.path.join(INPUT_DIR,  "explicitgemini.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "SemanticSimilarity_gemini_explicit.tsv"),
)

In [ ]:
run_sem_for_file(
    input_path  = os.path.join(INPUT_DIR,  "explicitgpt.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "SemanticSimilarity_gpt_explicit.tsv"),
)

In [ ]:
run_sem_for_file(
    input_path  = os.path.join(INPUT_DIR,  "explicitllama.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "SemanticSimilarity_llama_explicit.tsv"),
)

In [ ]:
run_sem_for_file(
    input_path  = os.path.join(INPUT_DIR,  "explicitqwen.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "SemanticSimilarity_qwen_explicit.tsv"),
)

In [ ]:
run_sem_for_file(
    input_path  = os.path.join(INPUT_DIR,  "implicitclaude.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "SemanticSimilarity_claude_implicit.tsv"),
)

In [ ]:
run_sem_for_file(
    input_path  = os.path.join(INPUT_DIR,  "implicitgemini.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "SemanticSimilarity_gemini_implicit.tsv"),
)

In [ ]:
run_sem_for_file(
    input_path  = os.path.join(INPUT_DIR,  "implicitgpt.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "SemanticSimilarity_gpt_implicit.tsv"),
)

In [ ]:
run_sem_for_file(
    input_path  = os.path.join(INPUT_DIR,  "implicitllama.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "SemanticSimilarity_llama_implicit.tsv"),
)

In [ ]:
run_sem_for_file(
    input_path  = os.path.join(INPUT_DIR,  "implicitqwen.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "SemanticSimilarity_qwen_implicit.tsv"),
)